# Model Training — Employee Attrition

This notebook compares **seven classification models** for employee attrition:

1. Logistic Regression
2. Random Forest
3. Support Vector Machine (SVM)
4. Gradient Boosting
5. Decision Tree
6. K-Nearest Neighbors (KNN)
7. Gaussian Naive Bayes

The dataset is imbalanced, so the notebook reports several metrics rather than using accuracy alone. Models are ranked using a documented **weighted cross-validation score** calculated only from the training data. The held-out test set is used for final evaluation, not model selection.

The saved pipeline is intended to be used by `backend/main.py`. Run all cells from top to bottom after making sure the notebook uses a compatible scikit-learn environment.

In [12]:
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
N_SPLITS = 5

In [13]:
data_candidates = [
    Path("../data/employee_attrition_selected.csv"),
    Path("backend/data/employee_attrition_selected.csv"),
    Path("/mnt/data/employee_attrition_selected.csv"),
]

data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "employee_attrition_selected.csv not found. "
        "Run this notebook from backend/notebooks or update data_candidates."
    )

df = pd.read_csv(data_path)

if "Attrition" not in df.columns:
    raise ValueError("The dataset must contain the target column 'Attrition'.")

df = df.dropna(subset=["Attrition"]).copy()
X = df.drop(columns=["Attrition"])
y = df["Attrition"].map({"Yes": 1, "No": 0})

if y.isna().any():
    raise ValueError("Attrition must contain only 'Yes' and 'No' values.")

y = y.astype(int)

print("Dataset path:", data_path.resolve())
print("Dataset shape:", df.shape)
print("Number of input features:", X.shape[1])
print("\nTarget distribution:")
print(df["Attrition"].value_counts())
print("\nTarget proportions:")
print(df["Attrition"].value_counts(normalize=True).round(4))
display(df.head())

Dataset path: C:\Users\kanga\Downloads\employee-attrition-prediction\backend\data\employee_attrition_selected.csv
Dataset shape: (1470, 33)
Number of input features: 32

Target distribution:
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Target proportions:
Attrition
No     0.8388
Yes    0.1612
Name: proportion, dtype: float64


,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,...,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,CompanyExperienceRatio,PromotionFrequency,IncomePerYearExperience,SatisfactionScore,RoleTenureRatio,Attrition
0,41,Travel_Rarely,Sales,1,2,Life Sciences,2,Female,3,2,...,6,4,0,5,0.750000,0.000,749.125000,2.00,0.666667,Yes
1,49,Travel_Frequently,Research & Development,8,1,Life Sciences,3,Male,2,2,...,10,7,1,7,1.000000,0.100,513.000000,3.00,0.700000,No
2,37,Travel_Rarely,Research & Development,2,2,Other,4,Male,2,1,...,0,0,0,0,0.000000,0.000,298.571429,3.00,0.000000,Yes
3,33,Travel_Frequently,Research & Development,3,4,Life Sciences,4,Female,3,1,...,8,7,3,0,1.000000,0.375,363.625000,3.25,0.875000,No
4,27,Travel_Rarely,Research & Development,2,1,Medical,1,Male,3,1,...,2,2,2,2,0.333333,1.000,578.000000,2.50,1.000000,No


In [14]:
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object", "category", "bool"]).columns.tolist()

print("Categorical features:", categorical_features)
print("\nNumerical features:", numeric_features)

Categorical features: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Numerical features: ['Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'CompanyExperienceRatio', 'PromotionFrequency', 'IncomePerYearExperience', 'SatisfactionScore', 'RoleTenureRatio']


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining target distribution:")
print(y_train.value_counts())
print("\nTest target distribution:")
print(y_test.value_counts())

Training samples: 1176
Testing samples: 294

Training target distribution:
Attrition
0    986
1    190
Name: count, dtype: int64

Test target distribution:
Attrition
0    247
1     47
Name: count, dtype: int64


## Preprocessing

- Numerical features are standardized using `StandardScaler`.
- Categorical features are one-hot encoded using `OneHotEncoder`.
- `handle_unknown="ignore"` allows the pipeline to handle categories that were not present during fitting.
- The preprocessing is inside each model's `Pipeline`, so during cross-validation it is fitted only on that fold's training partition.

Scaling is particularly important for Logistic Regression, SVM, and KNN. Tree models generally do not need scaling, but the same pipeline structure keeps data handling consistent.

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
    remainder="drop",
)

print("Preprocessor configured.")

Preprocessor configured.


## Define all seven classification models

These are baseline configurations, not exhaustive hyperparameter searches. Class weighting is used where supported to account for the imbalanced target. Gradient Boosting receives balanced sample weights during fitting. KNN uses distance-weighted neighbors as a reasonable baseline.

In [17]:
model_estimators = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "SVM": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE,
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
    ),
    "Gaussian Naive Bayes": GaussianNB(),
}

# Each estimator gets its own pipeline. The preprocessor is fitted inside
# each training fold during cross-validation.
models = {
    name: Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", estimator),
        ]
    )
    for name, estimator in model_estimators.items()
}

print(f"Models to compare: {len(models)}")
for index, name in enumerate(models, start=1):
    print(f"{index}. {name}")

Models to compare: 7
1. Logistic Regression
2. Random Forest
3. SVM
4. Gradient Boosting
5. Decision Tree
6. KNN
7. Gaussian Naive Bayes


## Compare models using five-fold stratified cross-validation

The overall score is defined before comparing results:

- F1-score: **30%**
- PR-AUC / Average Precision: **20%**
- Recall: **15%**
- ROC-AUC: **15%**
- Balanced accuracy: **10%**
- Precision: **5%**
- Accuracy: **5%**

This weighting prioritizes the detection of the minority attrition class while still considering overall classification performance. It is a project-specific ranking score, not a standard ML metric. The winning model is selected from cross-validation results on the training set only.

For Gradient Boosting, balanced sample weights are calculated separately inside each training fold.

In [18]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


def fit_pipeline(model, model_name, X_fit, y_fit):
    """Fit a pipeline, applying balanced sample weights to Gradient Boosting."""
    if model_name == "Gradient Boosting":
        counts = y_fit.value_counts()
        sample_weights = y_fit.map({
            class_label: len(y_fit) / (2.0 * count)
            for class_label, count in counts.items()
        }).to_numpy()

        model.fit(
            X_fit,
            y_fit,
            model__sample_weight=sample_weights
        )
    else:
        model.fit(X_fit, y_fit)

    return model


def calculate_metrics(y_true, y_pred, y_prob=None):
    """Calculate the four required classification metrics."""
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1 Score": f1_score(
            y_true, y_pred, zero_division=0
        ),
    }


cv_metric_names = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
]

cv_results = {}
fitted_models = {}

for model_name, base_model in models.items():
    fold_scores = {
        metric: [] for metric in cv_metric_names
    }

    for train_indices, validation_indices in cv.split(X_train, y_train):
        X_fold_train = X_train.iloc[train_indices]
        X_fold_valid = X_train.iloc[validation_indices]

        y_fold_train = y_train.iloc[train_indices]
        y_fold_valid = y_train.iloc[validation_indices]

        fold_model = clone(base_model)

        fit_pipeline(
            fold_model,
            model_name,
            X_fold_train,
            y_fold_train
        )

        fold_predictions = fold_model.predict(X_fold_valid)

        fold_metrics = calculate_metrics(
            y_fold_valid,
            fold_predictions
        )

        for metric in cv_metric_names:
            fold_scores[metric].append(fold_metrics[metric])

    mean_cv_metrics = {
        metric: float(np.mean(scores))
        for metric, scores in fold_scores.items()
    }

    cv_results[model_name] = mean_cv_metrics

    # Refit the model using the complete training set.
    final_model = clone(base_model)

    fit_pipeline(
        final_model,
        model_name,
        X_train,
        y_train
    )

    fitted_models[model_name] = final_model

    print(f"\nModel: {model_name}")

    for metric in cv_metric_names:
        print(f"{metric}: {mean_cv_metrics[metric]:.4f}")


# Display the cross-validation comparison table.
cv_results_df = pd.DataFrame(cv_results).T
cv_results_df.index.name = "Model"

display(cv_results_df.round(4))


Model: Logistic Regression
Accuracy: 0.7729
Precision: 0.3934
Recall: 0.7158
F1 Score: 0.5055

Model: Random Forest
Accuracy: 0.8673
Precision: 0.9250
Recall: 0.2053
F1 Score: 0.3257

Model: SVM
Accuracy: 0.8384
Precision: 0.5056
Recall: 0.6000
F1 Score: 0.5458

Model: Gradient Boosting
Accuracy: 0.8444
Precision: 0.5171
Recall: 0.5316
F1 Score: 0.5226

Model: Decision Tree
Accuracy: 0.7968
Precision: 0.3734
Recall: 0.3737
F1 Score: 0.3706

Model: KNN
Accuracy: 0.8529
Precision: 0.7197
Recall: 0.1579
F1 Score: 0.2541

Model: Gaussian Naive Bayes
Accuracy: 0.6793
Precision: 0.3183
Recall: 0.7526
F1 Score: 0.4402


,Accuracy,Precision,Recall,F1 Score
Model,,,,
Logistic Regression,0.7729,0.3934,0.7158,0.5055
Random Forest,0.8673,0.9250,0.2053,0.3257
SVM,0.8384,0.5056,0.6000,0.5458
Gradient Boosting,0.8444,0.5171,0.5316,0.5226
Decision Tree,0.7968,0.3734,0.3737,0.3706
KNN,0.8529,0.7197,0.1579,0.2541
Gaussian Naive Bayes,0.6793,0.3183,0.7526,0.4402


## Evaluate each fitted model on the untouched test set

Test-set results are reported for transparency. **They are not used to select the winning model.** The selected model is ranked using the cross-validation overall score calculated above.

In [19]:
results = []
predictions = {}

for model_name, model in fitted_models.items():
    test_predictions = model.predict(X_test)
    predictions[model_name] = test_predictions

    test_metrics = calculate_metrics(
        y_test, test_predictions, None
    )

    row = {
        "Model": model_name,
        "Accuracy": test_metrics["Accuracy"],
        "Precision": test_metrics["Precision"],
        "Recall": test_metrics["Recall"],
        "F1 Score": test_metrics["F1 Score"],
    }

    results.append(row)

results_df = pd.DataFrame(results).sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

,Model,Accuracy,Precision,Recall,F1 Score
0,SVM,0.8197,0.4516,0.5957,0.5138
1,Logistic Regression,0.7721,0.3864,0.7234,0.5037
2,Gradient Boosting,0.7959,0.3818,0.4468,0.4118
3,Gaussian Naive Bayes,0.6871,0.2897,0.6596,0.4026
4,Decision Tree,0.7619,0.2444,0.2340,0.2391
5,Random Forest,0.8435,0.5556,0.1064,0.1786
6,KNN,0.8231,0.3077,0.0851,0.1333


## Select the best model

The selected model is the one with the highest **Overall CV Score**. Cross-validation metrics are used for ranking; the held-out test metrics are for final reporting only. This avoids selecting a model simply because it happened to perform well on the test set.

In [20]:
# Select the best model based on cross-validation F1 score.
best_model_name = max(
    cv_results,
    key=lambda name: cv_results[name]["F1 Score"]
)

best_model = fitted_models[best_model_name]

print("Model Ranking (Cross-Validation):")

ranking = sorted(
    cv_results.items(),
    key=lambda item: item[1]["F1 Score"],
    reverse=True
)

for index, (model_name, metrics) in enumerate(ranking, start=1):
    print(
        f"{index}. {model_name} | "
        f"Accuracy: {metrics['Accuracy']:.4f} | "
        f"Precision: {metrics['Precision']:.4f} | "
        f"Recall: {metrics['Recall']:.4f} | "
        f"F1 Score: {metrics['F1 Score']:.4f}"
    )

print("\nSelected Final Model:", best_model_name)
print("Selection Criterion: Highest cross-validation F1 score.")

Model Ranking (Cross-Validation):
1. SVM | Accuracy: 0.8384 | Precision: 0.5056 | Recall: 0.6000 | F1 Score: 0.5458
2. Gradient Boosting | Accuracy: 0.8444 | Precision: 0.5171 | Recall: 0.5316 | F1 Score: 0.5226
3. Logistic Regression | Accuracy: 0.7729 | Precision: 0.3934 | Recall: 0.7158 | F1 Score: 0.5055
4. Gaussian Naive Bayes | Accuracy: 0.6793 | Precision: 0.3183 | Recall: 0.7526 | F1 Score: 0.4402
5. Decision Tree | Accuracy: 0.7968 | Precision: 0.3734 | Recall: 0.3737 | F1 Score: 0.3706
6. Random Forest | Accuracy: 0.8673 | Precision: 0.9250 | Recall: 0.2053 | F1 Score: 0.3257
7. KNN | Accuracy: 0.8529 | Precision: 0.7197 | Recall: 0.1579 | F1 Score: 0.2541

Selected Final Model: SVM
Selection Criterion: Highest cross-validation F1 score.


## Confusion matrices and classification reports

The confusion matrix and report help explain false positives and false negatives. For attrition prediction, false negatives (employees who leave but were predicted to stay) are particularly important to review.

In [21]:
for model_name, model in fitted_models.items():
    test_predictions = model.predict(X_test)

    test_metrics = calculate_metrics(
        y_test,
        test_predictions
    )

    print("\n" + "=" * 45)
    print(model_name)
    print("=" * 45)

    print(f"Accuracy:  {test_metrics['Accuracy']:.4f}")
    print(f"Precision: {test_metrics['Precision']:.4f}")
    print(f"Recall:    {test_metrics['Recall']:.4f}")
    print(f"F1 Score:  {test_metrics['F1 Score']:.4f}")


Logistic Regression
Accuracy:  0.7721
Precision: 0.3864
Recall:    0.7234
F1 Score:  0.5037

Random Forest
Accuracy:  0.8435
Precision: 0.5556
Recall:    0.1064
F1 Score:  0.1786

SVM
Accuracy:  0.8197
Precision: 0.4516
Recall:    0.5957
F1 Score:  0.5138

Gradient Boosting
Accuracy:  0.7959
Precision: 0.3818
Recall:    0.4468
F1 Score:  0.4118

Decision Tree
Accuracy:  0.7619
Precision: 0.2444
Recall:    0.2340
F1 Score:  0.2391

KNN
Accuracy:  0.8231
Precision: 0.3077
Recall:    0.0851
F1 Score:  0.1333

Gaussian Naive Bayes
Accuracy:  0.6871
Precision: 0.2897
Recall:    0.6596
F1 Score:  0.4026


## Save the selected model and comparison results

The final section writes deployment artifacts under `backend/models/`. Run the entire notebook from top to bottom before using these files in the application.

In [22]:
# Save deployment-ready artifacts
from sklearn.inspection import permutation_importance

artifact_dir = Path("../models")
artifact_dir.mkdir(parents=True, exist_ok=True)

# 1. Save the selected fitted model pipeline.
joblib.dump(best_model, artifact_dir / "attrition_model.pkl")

# 2. Save the model comparison table.
results_df.to_csv(
    artifact_dir / "model_comparison.csv",
    index=False
)

# 3. Calculate predictions for the selected model.
best_predictions = best_model.predict(X_test)
best_cm = confusion_matrix(
    y_test, best_predictions, labels=[0, 1]
).tolist()

best_metrics = calculate_metrics(y_test, best_predictions)

# 4. Calculate permutation feature importance.
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

feature_importance = [
    {
        "feature": feature,
        "importance": round(float(importance), 6),
    }
    for feature, importance in zip(
        X_test.columns, perm.importances_mean
    )
]

feature_importance.sort(
    key=lambda item: item["importance"],
    reverse=True
)

# 5. Save metrics for the selected model.
selected_cv = cv_results[best_model_name]

metrics = {
    "model_used": best_model_name,
    "selected_model": best_model_name,

    "accuracy": round(float(best_metrics["Accuracy"]), 4),
    "precision": round(float(best_metrics["Precision"]), 4),
    "recall": round(float(best_metrics["Recall"]), 4),
    "f1_score": round(float(best_metrics["F1 Score"]), 4),

    "cv_metrics": {
        metric.lower().replace(" ", "_"): round(
            float(selected_cv[metric]), 4
        )
        for metric in cv_metric_names
    },

    "confusion_matrix": best_cm,

    "comparison": [
        {
            "model": row["Model"],
            "accuracy": round(float(row["Accuracy"]), 4),
            "precision": round(float(row["Precision"]), 4),
            "recall": round(float(row["Recall"]), 4),
            "f1_score": round(float(row["F1 Score"]), 4),
        }
        for _, row in results_df.iterrows()
    ],

    "feature_importance": feature_importance,
    "selection_criterion": "Highest cross-validation F1 score",
    "feature_count": int(X.shape[1]),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "trained_at": datetime.now(timezone.utc).isoformat(),
}

with open(
    artifact_dir / "metrics.json", "w", encoding="utf-8"
) as file:
    json.dump(metrics, file, indent=4)


# 6. Save dashboard dataset statistics.
def grouped_counts_for_dashboard(frame, column):
    grouped = (
        frame.groupby([column, "Attrition"], dropna=False)
        .size()
        .unstack(fill_value=0)
    )

    rows = []

    for label, values in grouped.iterrows():
        yes = int(values.get("Yes", 0))
        no = int(values.get("No", 0))
        total = yes + no

        rows.append({
            "label": str(label),
            "yes": yes,
            "no": no,
            "total": total,
            "rate": round(yes / total * 100, 1) if total else 0,
        })

    return rows


total_employees = int(len(df))
employees_left = int((df["Attrition"] == "Yes").sum())
employees_stayed = int((df["Attrition"] == "No").sum())

target_distribution = [
    {
        "label": "Stayed",
        "count": employees_stayed,
        "percentage": round(
            employees_stayed / total_employees * 100, 1
        ),
    },
    {
        "label": "Left",
        "count": employees_left,
        "percentage": round(
            employees_left / total_employees * 100, 1
        ),
    },
]

income_bins = [0, 3000, 6000, 9000, 12000, float("inf")]
income_labels = ["Under 3k", "3k–6k", "6k–9k", "9k–12k", "12k+"]

income_groups = (
    df.assign(
        income_bucket=pd.cut(
            df["MonthlyIncome"],
            bins=income_bins,
            labels=income_labels,
            right=False,
        )
    )
    .groupby(["income_bucket", "Attrition"], observed=False)
    .size()
    .unstack(fill_value=0)
)

income_by_attrition = [
    {
        "bucket": str(label),
        "yes": int(values.get("Yes", 0)),
        "no": int(values.get("No", 0)),
    }
    for label, values in income_groups.iterrows()
]

dataset_stats = {
    "total_employees": total_employees,
    "employees_left": employees_left,
    "employees_stayed": employees_stayed,
    "attrition_rate": round(
        employees_left / total_employees * 100, 1
    ),
    "target_distribution": target_distribution,
    "by_department": grouped_counts_for_dashboard(df, "Department"),
    "by_job_role": grouped_counts_for_dashboard(df, "JobRole"),
    "by_overtime": grouped_counts_for_dashboard(df, "OverTime"),
    "income_by_attrition": income_by_attrition,
}

with open(
    artifact_dir / "dataset_stats.json", "w", encoding="utf-8"
) as file:
    json.dump(dataset_stats, file, indent=4)


print("\n" + "=" * 50)
print("DEPLOYMENT ARTIFACTS CREATED")
print("=" * 50)
print("Selected model:", best_model_name)
print("Accuracy:", round(best_metrics["Accuracy"], 4))
print("Precision:", round(best_metrics["Precision"], 4))
print("Recall:", round(best_metrics["Recall"], 4))
print("F1 Score:", round(best_metrics["F1 Score"], 4))
print("\nSaved files:")
print("Model:", artifact_dir / "attrition_model.pkl")
print("Metrics:", artifact_dir / "metrics.json")
print("Dataset stats:", artifact_dir / "dataset_stats.json")
print("Comparison:", artifact_dir / "model_comparison.csv")


DEPLOYMENT ARTIFACTS CREATED
Selected model: SVM
Accuracy: 0.8197
Precision: 0.4516
Recall: 0.5957
F1 Score: 0.5138

Saved files:
Model: ..\models\attrition_model.pkl
Metrics: ..\models\metrics.json
Dataset stats: ..\models\dataset_stats.json
Comparison: ..\models\model_comparison.csv
